In [19]:
import pandas as pd
import numpy as np
import json
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, roc_auc_score

pd.set_option("display.max_columns", None)

In [20]:
FEATURES_PATH = "../data/features"

X = pd.read_csv(f"{FEATURES_PATH}/X_features.csv")
y = pd.read_csv(f"{FEATURES_PATH}/y_target.csv").iloc[:, 0].map({"No": 0, "Yes": 1})

with open(f"{FEATURES_PATH}/column_metadata.json") as f:
    col_meta = json.load(f)

print(f"Features: {X.shape}")
print(f"Target distribution: {y.value_counts().to_dict()}")

Features: (500, 25)
Target distribution: {0: 445, 1: 55}


## 1. Train/Test Split

Use stratified split to preserve the 89/11 class imbalance in both sets. This ensures our evaluation metrics are meaningful.

In [21]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Train: {X_train.shape[0]} samples")
print(f"Test: {X_test.shape[0]} samples")
print(f"Train target dist: {y_train.value_counts().to_dict()}")
print(f"Test target dist: {y_test.value_counts().to_dict()}")

Train: 400 samples
Test: 100 samples
Train target dist: {0: 356, 1: 44}
Test target dist: {0: 89, 1: 11}


## 2. Logistic Regression Baseline

Logistic Regression is fast, interpretable, and outputs probabilities — which we need for risk bucketing later. We use `max_iter=1000` to ensure convergence.

In [22]:
numeric_cols = col_meta["numeric_cols"]
categorical_cols = col_meta["categorical_cols"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", make_pipeline(
            SimpleImputer(strategy="median"),
            StandardScaler()
        ), numeric_cols),
        ("cat", make_pipeline(
            SimpleImputer(strategy="most_frequent"),
            OneHotEncoder(handle_unknown="ignore", sparse_output=False)
        ), categorical_cols)
    ]
)

model = make_pipeline(
    preprocessor,
    LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
)
model.fit(X_train, y_train)
print("LogisticRegression trained with class_weight='balanced'")

LogisticRegression trained with class_weight='balanced'


## 3. Evaluation

We evaluate on precision, recall, F1, and ROC-AUC — not accuracy. With 89% of employees in the majority class, accuracy would be misleading.

In [23]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=["No", "Yes"]))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")

Classification Report:
              precision    recall  f1-score   support

          No       0.93      0.80      0.86        89
         Yes       0.25      0.55      0.34        11

    accuracy                           0.77       100
   macro avg       0.59      0.67      0.60       100
weighted avg       0.86      0.77      0.80       100

ROC-AUC: 0.7916


## 4. Feature Importance

Logistic Regression coefficients show which features push the prediction toward attrition (positive) or retention (negative).

In [24]:
# Get feature names after preprocessing
ohe_features = model["columntransformer"].named_transformers_["cat"]["onehotencoder"].get_feature_names_out(categorical_cols)
all_features = list(numeric_cols) + list(ohe_features)

coef_df = pd.DataFrame({
    "Feature": all_features,
    "Coefficient": model["logisticregression"].coef_[0]
}).sort_values("Coefficient", key=abs, ascending=False)

print("Top 10 features by absolute coefficient:")
print(coef_df.head(10).to_string(index=False))

Top 10 features by absolute coefficient:
                Feature  Coefficient
     JobRole_Accountant     1.559929
JobRole_Sales Executive     1.492757
        JobRole_Auditor    -1.374716
 LastPromotionYear_2018    -1.337133
 LastPromotionYear_2010     1.289770
JobRole_Account Manager    -1.271683
 LastPromotionYear_2016    -1.245346
 LastPromotionYear_2012    -1.091256
 LastPromotionYear_2017     1.038783
  OvertimeHoursPerMonth     1.009850


## Summary

This is our baseline. Key points:
- **Preprocessing in pipeline**: Median imputation + one-hot encoding + scaling all happen inside the pipeline, preventing data leakage
- **class_weight='balanced'**: Compensates for the 89/11 class imbalance
- A random guess would yield ROC-AUC of 0.50 — we need significantly higher

### Limitation: Small Test Set
With only 11 positive samples in the test set, our metrics have high variance. A single misclassification changes Recall by 9 percentage points. The cross-validation results (using all 500 samples) provide more stable estimates.